## Feature Engineering

Feature engineering transforms the cleaned and validated transaction data into predictive variables suitable for machine-learning models.

The objective is to create features that capture transaction size, customer behaviour, transaction velocity, account characteristics, time patterns, and transaction context while avoiding information that would only become available after the fraud outcome is known.

In [1]:
# import libraries
import pandas as pd
import numpy as np

In [2]:
# Load the EDA-ready dataset
df = pd.read_csv(
    r"C:\Users\telvi\Downloads\AMDARI\finlora_finance_flow\data\processed_data\finlora_eda_ready.csv"
)

In [3]:
print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (126000, 28)
    transaction_id      account_id account_type        kyc_tier  \
0  FLR250216186265  FLR-ACC-100000   individual  tier3_enhanced   
1  FLR250423143305  FLR-ACC-100000   individual  tier3_enhanced   
2  FLR250424154897  FLR-ACC-100000   individual  tier3_enhanced   
3  FLR250428101772  FLR-ACC-100000   individual  tier3_enhanced   
4  FLR250628102827  FLR-ACC-100000   individual  tier3_enhanced   

             timestamp day_of_week  hour_of_day  \
0  2025-02-16 14:55:35      Sunday           14   
1  2025-04-23 13:35:40   Wednesday           13   
2  2025-04-24 12:30:34    Thursday           12   
3  2025-04-28 14:41:59      Monday           14   
4  2025-06-28 16:15:47    Saturday           16   

                             description         merchant_name  \
0                    CNP PURCHASE - BOLT                  Bolt   
1                   WEB PURCHASE - SLACK                 Slack   
2  MOBILE PURCHASE - JUSTRITE SUPERSTORE   Justrite Superstore  

In [4]:
# Check data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 28 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   transaction_id               126000 non-null  object 
 1   account_id                   126000 non-null  object 
 2   account_type                 126000 non-null  object 
 3   kyc_tier                     126000 non-null  object 
 4   timestamp                    126000 non-null  object 
 5   day_of_week                  126000 non-null  object 
 6   hour_of_day                  126000 non-null  int64  
 7   description                  126000 non-null  object 
 8   merchant_name                126000 non-null  object 
 9   merchant_category            126000 non-null  object 
 10  channel                      126000 non-null  object 
 11  amount                       126000 non-null  float64
 12  currency                     126000 non-null  object 
 13 

In [6]:
# Convert timestamp from object to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Convert account creation date from object to datetime
df["account_created_date"] = pd.to_datetime(df["account_created_date"])

In [7]:
# Check the data types after conversion
df[["timestamp", "account_created_date"]].dtypes

timestamp               datetime64[ns]
account_created_date    datetime64[ns]
dtype: object

Create transaction amount features

Our EDA showed that amount is highly right-skewed, with very large transaction values.

A logarithmic transformation can reduce the influence of extreme values.

However, because our dataset contains negative transaction amounts, we should use log1p(abs(amount)) rather than directly applying log1p(amount).

In [8]:
# Log transformation of transaction amount
df["log_amount"] = np.log1p(df["amount"].abs())

# Log transformation of amount-to-average ratio
df["log_amount_to_avg_ratio"] = np.log1p(
    df["amount_to_avg_ratio"].abs()
)

print(
    df[
        [
            "amount",
            "log_amount",
            "amount_to_avg_ratio",
            "log_amount_to_avg_ratio"
        ]
    ].head()
)

     amount  log_amount  amount_to_avg_ratio  log_amount_to_avg_ratio
0  84233.00   11.341354                 9.21                 2.323368
1   9143.33    9.120889                 1.00                 0.693147
2  20639.00    9.934986                 2.26                 1.181727
3   3177.43    8.064143                 0.21                 0.190620
4  97839.52   11.491094                 1.97                 1.088562


Why these features?
amount captures the actual transaction value.
log_amount reduces the effect of extremely large transactions.
amount_to_avg_ratio captures how unusual the transaction is relative to the account's normal behaviour.
log_amount_to_avg_ratio makes the highly skewed ratio easier for Logistic Regression to model.

## 4. Create time-based features

You already have hour_of_day and day_of_week, but we can create cyclical representations because time is naturally circular.

For example, 23:00 and 00:00 are close together even though numerically they are far apart.

In [12]:
# Convert day of week to numerical values
day_mapping = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}

df["day_of_week_num"] = df["day_of_week"].map(day_mapping)

# Cyclical encoding for day of week
df["day_sin"] = np.sin(
    2 * np.pi * df["day_of_week_num"] / 7
)

df["day_cos"] = np.cos(
    2 * np.pi * df["day_of_week_num"] / 7
)

In [14]:
# Cyclical encoding for hour of day
df["hour_sin"] = np.sin(
    2 * np.pi * df["hour_of_day"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour_of_day"] / 24
)

In [15]:
# verify 
df[[
    "day_of_week",
    "day_of_week_num",
    "day_sin",
    "day_cos",
    "hour_of_day",
    "hour_sin",
    "hour_cos"
]].head()

,day_of_week,day_of_week_num,day_sin,day_cos,hour_of_day,hour_sin,hour_cos
0,Sunday,6,-0.781831,0.623490,14,-5.000000e-01,-0.866025
1,Wednesday,2,0.974928,-0.222521,13,-2.588190e-01,-0.965926
2,Thursday,3,0.433884,-0.900969,12,1.224647e-16,-1.000000
3,Monday,0,0.000000,1.000000,14,-5.000000e-01,-0.866025
4,Saturday,5,-0.974928,-0.222521,16,-8.660254e-01,-0.500000


These features allow the Logistic Regression model to capture patterns across the boundaries between midnight and 23:00 or Sunday and Monday.

## 5. Create transaction behaviour features

You already have some particularly useful behavioural variables.

Let's create a few explicit indicators.

In [16]:
# High transaction amount relative to the account's normal behaviour
df["high_amount_ratio_flag"] = (
    df["amount_to_avg_ratio"] >= 3
).astype(int)

# High transaction velocity
df["high_velocity_flag"] = (
    df["transaction_velocity_1h"] >= 1
).astype(int)

# New device indicator
df["new_device_flag"] = (
    df["is_new_device"] == 1
).astype(int)

# Cross-border transaction indicator
df["cross_border_flag"] = (
    df["is_cross_border"] == 1
).astype(int)

## 6. Create account-age bands

Account age is currently continuous. We can also create a simple categorical representation.

In [17]:
# Create account age groups
df["account_age_group"] = pd.cut(
    df["account_age_days"],
    bins=[-1, 30, 90, 365, np.inf],
    labels=[
        "new",
        "recent",
        "established",
        "mature"
    ]
)

print(df["account_age_group"].value_counts())

account_age_group
mature         53892
established    34027
new            30979
recent          7102
Name: count, dtype: int64


This gives us:

new → 0–30 days
recent → 31–90 days
established → 91–365 days
mature → more than 365 days

## 7. Keep transaction velocity

We have now independently validated this feature:
Velocity differences: 0
Percentage matching: 100.0%

In [18]:
# Therefore, we retain:
df["transaction_velocity_1h"]

0         0
1         0
2         0
3         0
4         0
         ..
125995    0
125996    0
125997    0
125998    0
125999    0
Name: transaction_velocity_1h, Length: 126000, dtype: int64

## 8. Handle is_new_device
Earlier we found:

-1 = 2.39%
0  = 2.58%
1  = 5.85%
Because -1 appears to represent an unknown/unavailable state, we shouldn't simply convert it to 0 and pretend it means "not new."

Create an explicit unknown indicator:

In [19]:
# Flag unknown device status
df["device_status_unknown"] = (
    df["is_new_device"] == -1
).astype(int)

# Create a modelling-friendly new-device variable
df["new_device_flag"] = (
    df["is_new_device"] == 1
).astype(int)

This preserves the distinction between:

known existing device
new device
unknown device status

## 9. Decide what NOT to use
| Feature               | Reason                                             |
| --------------------- | -------------------------------------------------- |
| `transaction_id`      | Identifier, not predictive behaviour               |
| `account_id`          | Identifier; can cause memorisation                 |
| `account_holder_name` | Personally identifiable information and not needed |
| `device_id`           | High-cardinality identifier                        |
| `description`         | NLP is outside the current LR/RF scope             |
| `timestamp`           | Use engineered time features instead               |
| `status`              | Potential target leakage                           |

The status issue is especially important. A fraud model should ideally use information available at the time the transaction is being assessed, not information generated after processing or investigation.


In [20]:
# Display the new feature columns
new_features = [
    "log_amount",
    "log_amount_to_avg_ratio",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "high_amount_ratio_flag",
    "high_velocity_flag",
    "new_device_flag",
    "cross_border_flag",
    "device_status_unknown",
    "account_age_group"
]

display(df[new_features].head())

,log_amount,log_amount_to_avg_ratio,hour_sin,hour_cos,day_sin,day_cos,high_amount_ratio_flag,high_velocity_flag,new_device_flag,cross_border_flag,device_status_unknown,account_age_group
0,11.341354,2.323368,-5.000000e-01,-0.866025,-0.781831,0.623490,1,0,0,0,1,mature
1,9.120889,0.693147,-2.588190e-01,-0.965926,0.974928,-0.222521,0,0,0,0,1,mature
2,9.934986,1.181727,1.224647e-16,-1.000000,0.433884,-0.900969,0,0,0,0,0,mature
3,8.064143,0.190620,-5.000000e-01,-0.866025,0.000000,1.000000,0,0,0,0,1,mature
4,11.491094,1.088562,-8.660254e-01,-0.500000,-0.974928,-0.222521,0,0,0,0,0,mature


In [21]:
print("New dataset shape:", df.shape)

New dataset shape: (126000, 41)


In [22]:
# Save the feature-engineered dataset
features_path = (
    r"C:\Users\telvi\Downloads\AMDARI\finlora_finance_flow"
    r"\data\processed_data\finlora_features.csv"
)

df.to_csv(features_path, index=False)

In [23]:
print("Feature-engineered dataset saved successfully.")
print("Shape:", df.shape)
print("Location:", features_path)

Feature-engineered dataset saved successfully.
Shape: (126000, 41)
Location: C:\Users\telvi\Downloads\AMDARI\finlora_finance_flow\data\processed_data\finlora_features.csv
